In [1]:
import os
os.chdir('/projects/f_wj183_1/work/xutao/2025_epi_LLM/RePORTAI/')
from langchain_core.messages import HumanMessage
from graph.builder import build_graph
from utils.context import load_context
from llm_vllm import build_llm
!nvidia-smi

Sat Dec 13 13:58:25 2025       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.154.05             Driver Version: 535.154.05   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA L40S                    On  | 00000000:17:00.0 Off |                    0 |
| N/A   40C    P0              82W / 350W |  41834MiB / 46068MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [ ]:
# from langchain_openai import ChatOpenAI
# llm = ChatOpenAI(
#         base_url="http://localhost:8000/v1",
#         api_key="dummy",
#         model="meta-llama/Llama-3.1-8B-Instruct",
#     )

# from langchain_core.prompts import ChatPromptTemplate
# prompt = ChatPromptTemplate.from_messages([
#     ("system", "You are a helpful data science assistant."),
#     ("user", "{question}")
# ])
# chain = prompt | llm
# result = chain.invoke({"question": "What is PCA in machine learning?"})
# print(result.content)

In [2]:
path_to_data='/projects/f_wj183_1/work/xutao/2025_epi_LLM/simple_rag/data/'
db_path='/projects/f_wj183_1/work/xutao/2025_epi_LLM/RePORTAI_db/agent_memory.db'
model_name = 'meta-llama/Llama-3.1-8B-Instruct'
base_url = 'http://localhost:8000/v1'
max_new_token = 512
temperature = 0.1
top_p = 0.9
data_path = f'{path_to_data}/data.json'
schema_path = f'{path_to_data}/column.json'

In [14]:
df, schema = load_context(data_path, schema_path)
llm = build_llm(model_name, temperature, top_p, base_url)
app = build_graph(llm, df, schema, db_path=db_path)
thread_id = "chat-session-1"
user_input = input("You: ") # 'What is sex distribution' how many tb positive cases
state = {
            "messages": [HumanMessage(content=user_input)],
            "generated_code": None,
            "output": None,
            "error": None,
            "status": "idle",
        }
config = {"configurable": {"thread_id": thread_id}}
final_state = None
print("\nAgent:")


Agent:


In [15]:

for event in app.stream(state, config=config):
    node_name = list(event.keys())[0]
    node_state = event[node_name]
    print(f"\n--- NODE: {node_name} ---")

    # Some nodes emit None (e.g. checkpoint nodes)
    if not isinstance(node_state, dict):
        continue
    final_state = node_state
    # Checkpoint nodes
    if node_name.startswith("human_review"):
        code = node_state.get("generated_code")

        if code:
            print(f"\n--- HUMAN CHECKPOINT {node_name} ---")
            print("Generated code:\n")
            print(code)
            print("\nApprove the code? (y/edit)")

            action = input("> ")

            if action == "y":
                continue   # resume execution
            else:
                instruction = input(
                    "Describe how you want the code changed (not the code itself):\n> ")
                node_state["revision_instruction"] = instruction
                node_state["status"] = "needs_revision"

print('Langraph end')
# Print final answer
if final_state:
    if final_state.get("status") == "ok":
        print(final_state["output"])
    else:
        print("Error:", final_state.get("error"))


--- NODE: generate_code ---

--- NODE: human_review_before_run ---

--- HUMAN CHECKPOINT human_review_before_run ---
Generated code:

print(f'TB positive cases: {df["tb_status"].value_counts()["Active TB"]}')

Approve the code? (y/n/edit)

--- NODE: execute_code ---

--- NODE: human_review_final ---

--- HUMAN CHECKPOINT human_review_final ---
Generated code:

print(f'TB positive cases: {df["tb_status"].value_counts()["Active TB"]}')

Approve the code? (y/n/edit)
Langraph end
TB positive cases: 219

